# Tutorial 06: Custom Models and Preprocessing

**About this tutorial**
- **Time:** ~30 minutes
- **Prerequisites:** [Tutorial 03](03_core_phenotyping_workflow.ipynb), [Tutorial 05](05_extending_phenome_plugins.ipynb)
- **Key ideas:** Custom `ModelWrapper`, `preprocessing_fn`, `custom_transformations`, optional `preprocessing_description`

While PhenoMe comes with highly optimized defaults (like DINOv2), you may want to use a different base model, apply custom image transformations (like specific normalizations), or preprocess your images with classical computer vision algorithms before they hit the neural network.

In this tutorial, you will learn how to:
1. Write a custom `preprocessing_fn` to alter images (e.g., using NumPy/scikit-image).
2. Provide `custom_transformations` to override PhenoMe's default PyTorch transforms.
3. Create a custom `ModelWrapper` to use a standard ResNet18 instead of DINOv2.
4. Optionally pass `preprocessing_description` / `transformations_description` so the INFO processing pipeline log describes those callables.

## 1. Setup and Toy Dataset

First, we'll initialize PhenoMe and download the standard *E. coli* toy dataset.

In [ ]:
import torch
import numpy as np
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights
from phenome import PhenoMe, get_metadata_from_path
from phenome.utils import ModelWrapper
import requests
import zipfile
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

def download_toy_dataset(output_dir="./toy_dataset"):
    output_path = Path(output_dir)
    if (output_path / "Ecoli_images").exists():
        print(f"Dataset already exists at {output_path}")
        return output_path

    url = "https://upvehueus-my.sharepoint.com/:u:/g/personal/aitor_gonzalezm_ehu_eus/IQCBYEL7E_-NTZZeIsBhwwG8AbI9BucCBAgaPacYqXjDsA4?download=1"
    output_path.mkdir(exist_ok=True, parents=True)
    zip_path = output_path / "ecoli_dataset.zip"
    
    print("Downloading E. coli dataset...")
    r = requests.get(url, allow_redirects=True)
    with open(zip_path, "wb") as f:
        f.write(r.content)
    
    print("Extracting...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(output_path)
    zip_path.unlink()
    
    extracted_images = list(output_path.glob("**/Ecoli_images"))
    return extracted_images[0].parent if extracted_images else output_path

dataset_root = download_toy_dataset()
image_dir = dataset_root / "Ecoli_images"

Now we locate the images and extract metadata.

In [ ]:
metadata_fn = get_metadata_from_path(".../(condition)/(timepoint)/(filename).*")
file_df = pheno.find_files(str(image_dir), metadata_fn=metadata_fn)
print(f"Found {len(file_df)} images.")

## 2. Custom Preprocessing Function

Sometimes you need to alter the raw image arrays before they are passed into the PyTorch dataset. You can pass a `preprocessing_fn` to `process_images`.

This function must take a single NumPy array (`H, W, C` or `H, W`) and return a NumPy array. For instance, let's write a simple function that clips the top and bottom 1% of intensities to enhance contrast.

In [ ]:
def enhance_contrast(image: np.ndarray) -> np.ndarray:
    """Clip bottom 1% and top 1% intensities to enhance contrast."""
    p1, p99 = np.percentile(image, (1, 99))
    # Clip values
    clipped = np.clip(image, p1, p99)
    # Re-normalize to [0, 1] range to avoid issues
    if p99 > p1:
        clipped = (clipped - p1) / (p99 - p1)
    else:
        clipped = np.zeros_like(clipped)
        
    # Maintain the same data type if needed, or return float32
    return clipped.astype(np.float32)

## 3. Custom PyTorch Transformations

By default, PhenoMe resizes the image and converts it to a PyTorch tensor. However, some base models require highly specific standardizations. We can override the default transforms by supplying `custom_transformations`.

Let's prepare the standard ImageNet normalization required by ResNet.

In [ ]:
my_transforms = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),
    # Standard ImageNet normalization
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## 4. Custom Model Wrapper

To plug any arbitrary PyTorch model into PhenoMe, you subclass `ModelWrapper` and implement `_get_embeddings()`. The pipeline expects `_get_embeddings()` to take a `(Batch, Channels, Height, Width)` tensor and return a `(Batch, Features)` tensor.

Let's wrap a `ResNet18` from `torchvision`. We'll use the final output of the linear layer (logits/classifier embedding) as our features.

In [ ]:
class ResNet18Wrapper(ModelWrapper):
    def __init__(self, device):
        # Load a pretrained ResNet18
        model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        
        # Initialize the parent class which moves the model to the device and sets it to eval()
        super().__init__(model=model, device=device)

    def _get_embeddings(self, tensor: torch.Tensor) -> torch.Tensor:
        out = self.model(tensor)
        return out

# Instantiate our custom wrapper
wrapper = ResNet18Wrapper(device=device)

## 5. Putting It All Together

We can now call `process_images`, passing our custom wrapper, preprocessing function, and custom transforms. We'll use `force_rgb=True` since ResNet expects 3 channels.

In [ ]:
pheno.process_images(
    model_wrapper=wrapper,
    channel_mode="split",               # Split channels if dealing with multi-channel fluorescence
    preprocessing_fn=enhance_contrast,  # Run our NumPy contrast clipping first
    preprocessing_description="clip 1st/99th percentiles, then rescale to 0-1",
    custom_transformations=my_transforms, # Then apply our PyTorch transforms
    transformations_description="Resize 224 then ImageNet normalize",
    force_rgb=True,                     # Replicate 1-channel to 3-channels for ResNet
    checkpoint_path="resnet_results.h5" # Save the output embeddings to HDF5
)

## 6. Verify the Custom Output

Let's confirm that we extracted embeddings correctly using our ResNet model. ResNet18 outputs 1000-dimensional features.

In [ ]:
embeddings = pheno.get_embeddings()
print(f"Embeddings shape: {embeddings.shape}")

# You can now proceed with standard visualization and downstream tasks!
pheno.plot_pca(color_by="condition")

### Summary
You have fully customized the image ingestion pipeline! You applied manual image corrections, swapped the transformation logic, and substituted the backbone AI model for a custom ResNet variant. This pattern works exactly the same for any other model, whether it's a custom-trained CNN or a large vision transformer.